# 💼 The Analyst's Notebook · Part 3
### Eleven instruments, one afternoon

In Part 1 you compared two stocks with a crude price range. In Part 2 you replaced it with a proper volatility, built out of every daily return of a year, using loops and a function you wrote yourself.

The desk now wants that report for the **whole universe**: eleven instruments, ten years of daily prices. You are not going to write eleven loops.

## How to work through this

- Run the **quick load** cell first. It brings back what you found in Part 2 and loads the price table.
- Each question builds on the last, so keep them in order and keep your variables. Later questions use the names earlier ones created.
- Cells with `...` are blanks. The notebook runs cleanly even before you fill them in, so **Run all** is always safe.
- Hints and solutions are folded under each question. Work first, then check.

*Stuck for more than 15 minutes? Ask a friend, ask an AI for a hint (not the answer), or email me at `jobo@econ.au.dk`.*

---

## ⚙️ Quick load

The three packages, the price table, and what Part 2 left you. Run it and read what it prints.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CANDIDATE_DIRS = ["data", os.path.join("..", "data"), "."]
REPO_RAW_URL = None      # set this in Colab if the file is not next to the notebook


def data_path(filename):
    """Where the course CSV files are, wherever you happen to be running."""
    for folder in CANDIDATE_DIRS:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    if REPO_RAW_URL is not None:
        return REPO_RAW_URL + filename
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from the course folder, "
        f"upload the CSV into Colab, or set REPO_RAW_URL."
    )


# The whole universe: eleven instruments, 2015 to 2024, one row per stock per day
prices = pd.read_csv(data_path("prices.csv"), parse_dates=["date"])
TICKERS = sorted(prices["ticker"].unique())

# --- What Part 2 found, restated so you can build on it ---
part2_vol = {          # daily volatility in 2024, computed with your own loops
    "AAPL": 0.0141,
    "KO":   0.0080,
    "NVDA": 0.0330,
    "JNJ":  0.0094,
    "JPM":  0.0148,
}
part2_riskiest = "NVDA"
part2_calmest = "KO"

print("Loaded prices:", prices.shape[0], "rows x", prices.shape[1], "columns")
print("Instruments:", ", ".join(TICKERS))
print()
print("Part 2 recap: daily volatility in 2024, five stocks")
for ticker in part2_vol:
    print(f"  {ticker:5} {part2_vol[ticker]:.4f}")
print(f"  riskiest: {part2_riskiest}   calmest: {part2_calmest}")

---

### Q1 · Look before you trust it

The quick load read the file with `pd.read_csv(path, parse_dates=['date'])`. Never compute on a table you have not looked at, so find out:

- what type of data each column holds
- which tickers are in it, and how many rows each one has

In [ ]:
print(prices.dtypes)
print()

rows_per_ticker = ...
rows_per_ticker

<details>
<summary>💡 Hint 1</summary>

`prices.dtypes` is already written for you. Read what it says about `date`.

</details>

<details>
<summary>💡 Hint 2</summary>

For the row counts, group by ticker and ask how big each group is: `prices.groupby('ticker').size()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
print(prices.dtypes)
print()

rows_per_ticker = prices.groupby('ticker').size()
rows_per_ticker
```

Eleven tickers, 2,516 rows each, and no surprises in the types: real dates, text tickers, float prices. Equal row counts matter. If one stock had fewer days you would be comparing different periods without noticing.

*`parse_dates` is what made `date` a real date rather than text. Without it, none of the slicing by year in Q2 would work.*

</details>

---

### Q2 · One column per stock

The table is **tall**: one row per stock per day. For comparing stocks it is easier **wide**: one row per date, one column per ticker.

Reshape `prices` into `wide`, holding the closing prices. Then take 2024 out of it into `p24`.

In [ ]:
wide = ...
p24 = ...
p24

<details>
<summary>💡 Hint 1</summary>

`prices.pivot(index='date', columns='ticker', values='close')` does the reshaping.

</details>

<details>
<summary>💡 Hint 2</summary>

Because the index is dates, `wide.loc['2024']` gives you just that year.

</details>

<details>
<summary>✅ Solution</summary>

```python
wide = prices.pivot(index='date', columns='ticker', values='close')
p24 = wide.loc['2024']
p24.head()
```

`p24` has 252 rows and 11 columns: every trading day of 2024, with every instrument beside it. That is the shape you want whenever you are comparing things rather than storing them.

</details>

---

### Q3 · Apple again, the short way

In Part 2 you computed Apple's 2024 volatility with two loops and a function of your own. Do it again from `p24`, in two lines, and compare the answers.

$$\sigma = \text{the standard deviation of the daily returns}$$

In [ ]:
aapl_returns = ...
aapl_vol = ...

print('Part 3:', aapl_vol)
print('Part 2:', part2_vol['AAPL'])

<details>
<summary>💡 Hint 1</summary>

`p24['AAPL'].pct_change()` gives every daily return, with a `NaN` on the first day because there is no day before it.

</details>

<details>
<summary>💡 Hint 2</summary>

Then `.std()` on those returns. It skips the `NaN` for you.

</details>

<details>
<summary>✅ Solution</summary>

```python
aapl_returns = p24['AAPL'].pct_change()
aapl_vol = aapl_returns.std()

print('Part 3:', round(aapl_vol, 4))
print('Part 2:', part2_vol['AAPL'])
```

Both give **0.0141**. Two loops, an empty list and a function of your own, replaced by `.pct_change().std()`.

*Compare more decimals and they differ very slightly. Your Part 2 function divided by $n$; pandas divides by $n-1$ by default. On 251 returns that moves the fifth decimal, and neither is wrong.*

</details>

---

### Q4 · All eleven at once

Now the point of the whole session. Compute the 2024 daily volatility of **every** instrument into `vol`, sorted riskiest first.

In [ ]:
vol = ...
vol

<details>
<summary>💡 Hint 1</summary>

`p24.pct_change()` gives a whole table of returns, one column per instrument.

</details>

<details>
<summary>💡 Hint 2</summary>

`.std()` on that table gives one number per column, and `.sort_values(ascending=False)` puts the riskiest at the top.

</details>

<details>
<summary>✅ Solution</summary>

```python
vol = p24.pct_change().std().sort_values(ascending=False)
vol
```

Eleven answers, one line. Nvidia at **0.0331** is more than four times as volatile as the calmest thing in the table. Your five Part 2 numbers are all in here, unchanged.

</details>

---

### Q5 · Check it a second way

That was quick enough to be worth distrusting. Compute the same eleven numbers **without** the wide table: take 2024 out of `prices`, add a return column within each ticker, and take the standard deviation of each group.

Then look at whether the two routes agree.

In [ ]:
p2024 = prices[prices['date'].dt.year == 2024].copy()

p2024['ret'] = ...
vol_check = ...
vol_check

<details>
<summary>💡 Hint 1</summary>

`p2024.groupby('ticker')['close'].pct_change()` computes the return **inside** each stock. Grouping first is what stops a return being taken across two different companies.

</details>

<details>
<summary>💡 Hint 2</summary>

Then `p2024.groupby('ticker')['ret'].std()`, sorted the same way as `vol`.

</details>

<details>
<summary>✅ Solution</summary>

```python
p2024 = prices[prices['date'].dt.year == 2024].copy()

p2024['ret'] = p2024.groupby('ticker')['close'].pct_change()
vol_check = p2024.groupby('ticker')['ret'].std().sort_values(ascending=False)
vol_check
```

The same eleven numbers in the same order. Two different routes agreeing is the cheapest check you will ever run, and worth doing whenever one line has replaced forty.

*The `groupby` before `pct_change` is the part that matters. Without it, the first return of each stock would be computed from the last price of whichever stock sits above it in the table.*

</details>

---

### Q6 · Put it in the units the desk uses

Nobody quotes a daily volatility. Annualise `vol`, then store the riskiest and the calmest name.

$$\sigma_{\text{year}} = \sigma_{\text{day}} \times \sqrt{252}$$

In [ ]:
annual_vol = ...
riskiest = ...
calmest = ...

print(annual_vol)
print('riskiest:', riskiest, ' calmest:', calmest)

<details>
<summary>💡 Hint 1</summary>

`np.sqrt(252)` is the multiplier, and you can apply it to the whole Series at once.

</details>

<details>
<summary>💡 Hint 2</summary>

`vol` is already sorted riskiest first, so `annual_vol.index[0]` is the riskiest name and `annual_vol.index[-1]` is the calmest.

</details>

<details>
<summary>✅ Solution</summary>

```python
annual_vol = (vol * np.sqrt(252)).round(3)
riskiest = annual_vol.index[0]
calmest = annual_vol.index[-1]

print(annual_vol)
print('riskiest:', riskiest, ' calmest:', calmest)
```

Nvidia at about **53% a year**, against **13%** for SPY.

Look at what came bottom: **SPY**, the fund that holds the whole S&P 500. It is calmer than every individual stock in the table, including Coca-Cola, which was the calmest thing you had found in Part 2. That is diversification, and you have just measured it.

</details>

---

### Q7 · Package it, so you can reuse it

You have now written the same calculation three times. Turn it into a **function**: `volatility(prices_series, periods=252)`, which takes a column of prices and returns an annualised volatility.

Give it a docstring, then check it against Apple's number from Q3.

In [ ]:
def volatility(prices_series, periods=252):
    ...


# Once it works, uncomment this to check it against Q6:
# print('function:', volatility(p24['AAPL']))
# print('Q6      :', annual_vol['AAPL'])

<details>
<summary>💡 Hint 1</summary>

The body is one line: `.pct_change()`, then `.std()`, then times `np.sqrt(periods)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`return prices_series.pct_change().std() * np.sqrt(periods)`. The default `periods=252` means you can call it with one argument most of the time.

</details>

<details>
<summary>✅ Solution</summary>

```python
def volatility(prices_series, periods=252):
    """Annualised volatility of a price series."""
    return prices_series.pct_change().std() * np.sqrt(periods)

print('AAPL, function:', round(volatility(p24['AAPL']), 3))
print('AAPL, Q6      :', annual_vol['AAPL'])
```

The same **0.224**. The function is Session 2 work: `def`, a parameter, a default, a docstring and a `return`. Only the line inside it is new.

*Try `volatility(p24['AAPL'], periods=21)` for a monthly figure. That is what the default argument buys you: one function, several questions.*

</details>

---

### Q8 · A loop, a dictionary, and a Series

Now rebuild the whole ranking the Session 2 way: **loop** over `TICKERS` (the quick load made that list for you), call your function on each one, and store the answers in a **dictionary** keyed by ticker. Then turn the dictionary into a Series and check it against `annual_vol`.

In [ ]:
by_hand = {}
for ticker in TICKERS:
    ...

by_hand = ...
by_hand

<details>
<summary>💡 Hint 1</summary>

Inside the loop: `by_hand[ticker] = volatility(p24[ticker])`.

</details>

<details>
<summary>💡 Hint 2</summary>

Afterwards, `pd.Series(by_hand).sort_values(ascending=False)` turns the dictionary into the same object `annual_vol` is.

</details>

<details>
<summary>✅ Solution</summary>

```python
by_hand = {}
for ticker in TICKERS:
    by_hand[ticker] = volatility(p24[ticker])

by_hand = pd.Series(by_hand).sort_values(ascending=False)
by_hand.round(3)
```

The same eleven numbers in the same order as Q6.

Nothing in that cell is new: a loop, a function, a dictionary, all from Session 2. It is worth seeing that the one-line pandas version and the eleven-line loop give the same answer, because the loop is the one you can still write when the calculation gets too odd for a single method call.

</details>

---

### Q9 · Show the desk, do not tell it

Draw the ranking as horizontal bars: tickers down the side, annualised volatility across. Give it a title and an x-axis label.

Sort it the other way round first, so the longest bar ends up at the top of the picture.

In [ ]:
ranked = ...

fig, ax = plt.subplots(figsize=(9, 4))
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

`ranked = annual_vol.sort_values()` puts the smallest first, which draws from the bottom up.

</details>

<details>
<summary>💡 Hint 2</summary>

Then `ax.barh(ranked.index, ranked.values)`, `ax.set_title('...', loc='left')` and `ax.set_xlabel('...')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
ranked = annual_vol.sort_values()

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(ranked.index, ranked.values)
ax.set_title('Annualised volatility, 2024', loc='left')
ax.set_xlabel('standard deviation of daily returns, annualised')
plt.show()
```

One picture, and the ranking is obvious without reading a number. Sorting before you plot is what makes a bar chart worth looking at: an unsorted one makes the reader do the work.

</details>

---

### Q10 · What the riskiest one actually did

A volatility says how much something moved, not which way. Draw Nvidia's 2024 closing price as a line, with a title and a y-axis label, and see for yourself.

In [ ]:
nvda = ...

fig, ax = plt.subplots(figsize=(9, 3))
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

`nvda = p24['NVDA']` is the column you want.

</details>

<details>
<summary>💡 Hint 2</summary>

Then `ax.plot(nvda.index, nvda.values)`, a title and `ax.set_ylabel('price (USD)')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
nvda = p24['NVDA']

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(nvda.index, nvda.values)
ax.set_title('Nvidia (NVDA) closing price, 2024', loc='left')
ax.set_ylabel('price (USD)')
plt.show()
```

It nearly tripled: a total return of about **179%** over the year. So the riskiest thing in the table was also the most rewarding one, in this particular year. Risk is not a synonym for bad, which is exactly why the two questions have to be kept apart.

</details>

---

### Q11 · Do they move together?

A portfolio's risk depends not only on how much each stock moves, but on whether they all move **at the same time**.

For Apple and for Coca-Cola, count the days in 2024 when they moved in the **same direction** as the market (SPY). Two returns moved the same way when their product is positive.

In [ ]:
r24 = ...
aapl_same = ...
ko_same = ...

print('AAPL with the market:', aapl_same)
print('KO   with the market:', ko_same)

<details>
<summary>💡 Hint 1</summary>

`r24 = p24.pct_change().dropna()` gives every daily return with the first, empty row removed.

</details>

<details>
<summary>💡 Hint 2</summary>

`r24['AAPL'] * r24['SPY']` is positive on a day they moved together, so `(r24['AAPL'] * r24['SPY'] > 0).sum()` counts those days. It is the up-day counter you have been writing since Session 2.

</details>

<details>
<summary>✅ Solution</summary>

```python
r24 = p24.pct_change().dropna()
aapl_same = (r24['AAPL'] * r24['SPY'] > 0).sum()
ko_same = (r24['KO'] * r24['SPY'] > 0).sum()

print('trading days       :', len(r24))
print('AAPL with the market:', aapl_same)
print('KO   with the market:', ko_same)
```

Out of 251 days, Apple moved with the market on **176** of them (70%) and Coca-Cola on only **131** (52%), which is barely better than a coin toss.

That gap matters more than it looks. A stock that moves with everything else adds its full risk to a portfolio. One that goes its own way partly cancels out against the others, which is why the S&P 500 fund came bottom of your ranking in Q6.

</details>

---

### Q12 · From a number to a category

The desk does not want eleven decimals, it wants three buckets. Write `risk_label(annual_vol)` that returns `'high'` above 0.30, `'medium'` above 0.18 and `'low'` otherwise, then label every stock and collect the answers.

In [ ]:
def risk_label(vol_value):
    ...

labels = {}
for ticker in TICKERS:
    ...

labels = ...
labels

<details>
<summary>💡 Hint 1</summary>

The function is three branches: `if annual_vol > 0.30: return 'high'`, then `elif annual_vol > 0.18: return 'medium'`, then `else: return 'low'`.

</details>

<details>
<summary>💡 Hint 2</summary>

In the loop, `labels[ticker] = risk_label(annual_vol[ticker])`. Afterwards, `pd.Series(labels)` makes it a column you can add to a table.

</details>

<details>
<summary>✅ Solution</summary>

```python
def risk_label(vol_value):
    """Bucket an annualised volatility into high, medium or low."""
    if vol_value > 0.30:
        return 'high'
    elif vol_value > 0.18:
        return 'medium'
    else:
        return 'low'

labels = {}
for ticker in TICKERS:
    labels[ticker] = risk_label(annual_vol[ticker])

labels = pd.Series(labels)
labels
```

Nvidia alone is `high`. Disney, JPMorgan, Apple, Microsoft and Exxon are `medium`, and the other five are `low`.

You have just turned a measured number into a **category**, using an `if` chain from Session 2 and nothing else. Predicting a number is called regression; predicting a category like this one is called classification. Session 4 gives both of them their names.

</details>

---

### Q13 · The table a model would want

Build the report as one row per stock, with three columns describing its 2024: annualised volatility, average daily return, and the share of days it rose.

Then add your risk labels as a fourth column.

In [ ]:
vol_col = ...
mean_col = ...
up_col = ...

report = ...
report

<details>
<summary>💡 Hint 1</summary>

`r24` from Q11 already holds every daily return. The three columns are `r24.std() * np.sqrt(252)`, `r24.mean()` and `(r24 > 0).mean()`.

</details>

<details>
<summary>💡 Hint 2</summary>

`pd.DataFrame({'vol': vol_col, 'mean_ret': mean_col, 'up_share': up_col})` lines them up on the ticker labels. Add the labels with `report['risk'] = labels`.

</details>

<details>
<summary>✅ Solution</summary>

```python
vol_col = r24.std() * np.sqrt(252)
mean_col = r24.mean()
up_col = (r24 > 0).mean()

report = pd.DataFrame({'vol': vol_col, 'mean_ret': mean_col, 'up_share': up_col}).round(4)
report['risk'] = labels
report.sort_values('vol', ascending=False)
```

Eleven rows, four columns. This is the report, and it is also a shape worth recognising: **one row per thing you are studying, one column per fact about it**. Session 4 calls the rows *observations*, the numeric columns *features*, and whichever column you are trying to predict the *target*.

Everything in this course from here on arrives in that shape.

</details>

---

### Q14 · Would last year have told you?

Every number above describes 2024, a year that has already happened. The useful question is whether it would have helped you in advance.

Compute each stock's annualised volatility over **2015 to 2022**, and again over **2023 to 2024**, put them side by side, and sort by the early column.

In [ ]:
early = ...
late = ...

early_vol = ...
late_vol = ...

comparison = ...
comparison

<details>
<summary>💡 Hint 1</summary>

`wide.loc['2015':'2022']` and `wide.loc['2023':'2024']` cut the history in two, because the index is dates.

</details>

<details>
<summary>💡 Hint 2</summary>

Each volatility is `period.pct_change().std() * np.sqrt(252)`, and `pd.DataFrame({'early': early_vol, 'late': late_vol})` puts them side by side.

</details>

<details>
<summary>✅ Solution</summary>

```python
early = wide.loc['2015':'2022']
late = wide.loc['2023':'2024']

early_vol = early.pct_change().std() * np.sqrt(252)
late_vol = late.pct_change().std() * np.sqrt(252)

comparison = pd.DataFrame({'early': early_vol, 'late': late_vol})
comparison.sort_values('early', ascending=False).round(3)
```

Partly, and only partly. Nvidia is the most volatile in both periods, and Coca-Cola and SPY are among the calmest in both. But Apple falls from second riskiest to sixth, and Johnson & Johnson climbs from last.

Notice what you did there: you fitted your view on one stretch of time and checked it on a **later** one you had not looked at. Splitting by time rather than at random is not a detail. A randomly chosen test day would sit between two days you had already seen, and any method would look better than it is.

That gap between the two columns is the subject of the rest of this course. A quantity that carried over perfectly would need no model; one that carried over not at all could not be predicted. Volatility sits in between, which is exactly what makes it worth forecasting.

</details>

---

## 🧭 What you have now

The risk report is finished. In three sessions it has gone from two stocks and a price range, to eleven instruments measured properly, ranked, drawn, labelled, and checked three different ways.

Notice how little of it was new. The volatility is a function you wrote, called in a loop, storing into a dictionary, with an `if` chain deciding the labels. pandas did not replace any of that. It replaced the tedious parts: reading the file, lining up the dates, and doing the arithmetic eleven times.

**Carry these into Part 4:**

| what | where |
|:--|:--|
| the price table, tall and wide | `prices`, `wide`, `p24` |
| 2024 volatility, daily and annualised | `vol`, `annual_vol` |
| riskiest and calmest | `riskiest`, `calmest` |
| a reusable calculation | `volatility()`, `risk_label()` |
| the risk categories | `labels` |
| one row per stock, four columns | `report` |


## The question this cannot answer

Everything above is about a year that has already happened. It is **description**: an honest account of what these prices did in 2024.

The desk does not really want that. It wants to know what next month looks like. The moment you ask that, the shape of the problem changes:

- which past facts are you allowed to use, and which would be cheating?
- what exactly are you predicting, and how would you know if you were right?
- if a rule describes 2024 perfectly, is that good news or a warning?

You have already built the pieces. `report` in Q13 is a table of observations and features. `labels` in Q12 is a category you could try to predict. The split in Q14 is how you would find out honestly whether you could.

Next session is about those three questions. You will not fit a model. You will learn what a model is being asked to do, and why that is harder than it sounds.

**Part 4** turns this report into a prediction problem: observations, features, and a target.